In [ ]:
from essential.gpu_utils import select_best_gpus

select_best_gpus(n_gpus=1)

import pandas as pd
import plotnine as gg
import numpy as np

from essential.utils import load_regulondb_full

In [ ]:
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

# Loading regulonDB; Evo2/LLM embeddings

In [ ]:
!pip install biopython

In [ ]:
from Bio import Entrez
from Bio import SeqIO
import os

Entrez.email = "pboyeau@broadinstitute.org"
accession_id = "U00096.3"
db = "nuccore"  # NCBI Nucleotide database
rettype = "gbwithparts"  # GenBank format with all features
retmode = "text"

output_filename = f"/workspace/data/ecoli_evo2/{accession_id}_gene_sequences.fasta"
print(f"Downloading GenBank record for E. coli ({accession_id})...")

handle = Entrez.efetch(db=db, id=accession_id, rettype=rettype, retmode=retmode)
genome_record = SeqIO.read(handle, "genbank")
handle.close()

gene_metadata = []
for feature in genome_record.features:
    if feature.type == "gene":
        gene_name = feature.qualifiers.get("gene", ["unknown"])[0]
        locus_tag = feature.qualifiers.get("locus_tag", [""])[0]
        promoter_len = 500
        start = feature.location.start
        end = feature.location.end
        strand = feature.location.strand

        gene_metadata.append(
            {
                "gene_name": gene_name,
                "gene_name_lower": gene_name.lower(),
                "locus_tag": locus_tag,
                "promoter_len": promoter_len,
                "start": start,
                "end": end,
            }
        )
gene_metadata = pd.DataFrame(gene_metadata)

In [ ]:
evo_data = np.load("/workspace/data/ecoli_evo2/evo2_embeddings.npz")
embeddings = np.array(evo_data["matrix"])
gene_names_ = evo_data["genes"]
gene_names = [g.lower() for g in gene_names_]

gene_to_id = {g.lower(): i for i, g in enumerate(gene_names)}

In [ ]:
llm_data = np.load("/workspace/data/ecoli_llm/llm_embeddings.npz")
llm_embeddings = np.array(llm_data["embeddings"])
llm_genes = llm_data["genes"]
llm_genes = [g.lower() for g in llm_genes]
llm_gene_to_id = {g: i for i, g in enumerate(llm_genes)}

In [ ]:
intersect_genes = set(gene_to_id.keys()) & set(llm_gene_to_id.keys())
intersect_genes = list(intersect_genes)

new_gene_order = [gene_to_id[g] for g in intersect_genes]
new_llm_gene_order = [llm_gene_to_id[g] for g in intersect_genes]


embeddings = embeddings[new_gene_order]
llm_embeddings = llm_embeddings[new_llm_gene_order]

gene_to_id = {g: i for i, g in enumerate(intersect_genes)}
gene_names = intersect_genes

In [ ]:
# from sklearn.manifold import TSNE
# from sklearn.metrics.pairwise import cosine_similarity
# from sklearn.decomposition import PCA
# from sklearn.cluster import KMeans

# pca = PCA(n_components=100)
# pca_result = pca.fit_transform(llm_embeddings)
# tsne = TSNE(n_components=2, random_state=42)
# tsne_result = tsne.fit_transform(pca_result)
# plot_df = pd.DataFrame(tsne_result, columns=["x", "y"])

# (gg.ggplot(plot_df) + gg.aes(x="x", y="y") + gg.geom_point(size=0.1) + gg.theme_minimal())

In [ ]:
# from sklearn.manifold import TSNE
# from sklearn.metrics.pairwise import cosine_similarity
# from sklearn.decomposition import PCA
# from sklearn.cluster import KMeans

# # cosine_dist = 1 - cosine_similarity(embeddings)
# # cosine_dist[cosine_dist < 0] = 0
# # tsne = TSNE(n_components=2, random_state=42, metric="precomputed", init="random")
# # tsne_result = tsne.fit_transform(cosine_dist)

# pca = PCA(n_components=100)
# pca_result = pca.fit_transform(embeddings)
# tsne = TSNE(n_components=2, random_state=42)
# tsne_result = tsne.fit_transform(pca_result)

# cluster_assignment = KMeans(n_clusters=2, random_state=42).fit_predict(pca_result)

# plot_df = pd.DataFrame(tsne_result, columns=["x", "y"])
# plot_df["cluster"] = cluster_assignment

# (
#     gg.ggplot(plot_df)
#     + gg.aes(x="x", y="y", color="cluster")
#     + gg.geom_point(size=0.1)
#     + gg.theme_minimal()
# )

In [ ]:
ref_db = load_regulondb_full()

should_keep_row = []
admissible_regulators = ref_db["regulator_gene"].isin(gene_names)
admissible_targets = ref_db["target_gene"].isin(gene_names)
ref_db_ = ref_db[admissible_regulators & admissible_targets].copy()
ref_db_["regulator_id"] = ref_db_["regulator_gene"].map(gene_to_id)
ref_db_["target_id"] = ref_db_["target_gene"].map(gene_to_id)
id_pairs = ref_db_["regulator_id"].astype(str) + "_" + ref_db_["target_id"].astype(str)

print("Number of rows in the reference database: ", len(ref_db))
print("Number of rows in the filtered reference database: ", len(ref_db_))

unique_targets = np.array(ref_db["target_gene"].unique())
unique_regulators = np.array(ref_db["regulator_gene"].unique())

unique_genes = np.unique(np.concatenate([unique_targets, unique_regulators]))
print("Number of unique genes: ", len(unique_genes))
print("Number of unique targets: ", len(unique_targets))
print("Number of unique regulators: ", len(unique_regulators))

In [ ]:
# flagellar_df = pd.read_csv("/workspace/data/ecoli_evo2/flagellar_genes.csv")
# validation_targets = flagellar_df["gene"].values
# train_targets = [t for t in unique_targets if t not in validation_targets]

# Train-test splitting

### option A: splitting targets

In [ ]:
bins = np.linspace(0, gene_metadata["start"].max(), 100)

gene_metadata_ = gene_metadata.loc[lambda x: x["gene_name_lower"].isin(unique_targets)]
validation_targets = gene_metadata_.loc[lambda x: (x["start"] >= 2e6) & (x["start"] <= 3e6)][
    "gene_name_lower"
].values
train_targets = [t for t in unique_targets if t not in validation_targets]

validation_metadata = gene_metadata_[gene_metadata_["gene_name_lower"].isin(validation_targets)]
train_metadata = gene_metadata_[gene_metadata_["gene_name_lower"].isin(train_targets)]


plt.hist(train_metadata["start"], bins=bins, label="Train")
plt.hist(validation_metadata["start"], bins=bins, label="Validation")
plt.xlabel("Start position")
plt.legend()
plt.show()

In [ ]:
# validation_targets = np.random.choice(unique_targets, size=1000, replace=False)
# train_targets = [t for t in unique_targets if t not in validation_targets]

In [ ]:
train_interactions = ref_db_[
    ref_db_["target_gene"].isin(train_targets) & ref_db_["regulator_gene"].isin(gene_names)
]
validation_interactions = ref_db_[
    ref_db_["target_gene"].isin(validation_targets) & ref_db_["regulator_gene"].isin(gene_names)
]

print(len(train_interactions), len(validation_interactions))

## Option B: splitting regulators

In [ ]:
ref_db_["regulator_gene"].value_counts().head(50)

In [ ]:
heldout_regulators = ["crp"]

In [ ]:
train_interactions = ref_db_[
    ref_db_["target_gene"].isin(gene_names) & ~(ref_db_["regulator_gene"].isin(heldout_regulators))
]
validation_interactions = ref_db_[
    ref_db_["target_gene"].isin(gene_names) & (ref_db_["regulator_gene"].isin(heldout_regulators))
]

print(len(train_interactions), len(validation_interactions))

In [ ]:
embeddings.shape, llm_embeddings.shape

# Main analysis

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, auc
from typing import List, Dict, Any


class MLP(nn.Module):
    """A simple Multi-Layer Perceptron model."""

    hidden_dim: int

    @nn.compact
    def predict(self, emb_i, emb_j, train: bool):
        x = jnp.concatenate([emb_i, emb_j], axis=-1)
        x = nn.Dense(features=self.hidden_dim)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.Dense(features=1)(x)
        return x.squeeze(-1)

    @nn.compact
    def __call__(self, emb_i, emb_j, y, train: bool = True):
        y_pred = self.predict(emb_i, emb_j, train=train)
        loss = optax.sigmoid_binary_cross_entropy(y_pred, y)
        return loss


class CausalPredictor:
    """
    A class to train and evaluate a causal prediction model on gene interaction data.

    This class encapsulates data preparation, model initialization, training,
    and evaluation, making it easy to experiment with different embeddings and
    hyperparameters.
    """

    def __init__(
        self,
        embeddings: np.ndarray,
        train_interactions: pd.DataFrame,
        validation_interactions: pd.DataFrame,
        gene_names: List[str],
        hidden_dim: int = 4096,
        learning_rate: float = 1e-3,
        batch_size: int = 1024,
    ):
        """
        Initializes the CausalPredictor.

        Args:
            embeddings: A numpy array of gene embeddings.
            train_interactions: DataFrame with positive training interactions.
            validation_interactions: DataFrame with positive validation interactions.
            gene_names: A list of gene names, indexed by gene ID.
            hidden_dim: The hidden dimension of the MLP.
            learning_rate: The learning rate for the Adam optimizer.
            batch_size: The batch size for training and evaluation.
        """
        self.embeddings = embeddings
        self.train_interactions = train_interactions
        self.validation_interactions = validation_interactions
        self.gene_names = gene_names
        self.hidden_dim = hidden_dim
        self.learning_rate = learning_rate
        self.batch_size = batch_size

        self.model = MLP(hidden_dim=self.hidden_dim)
        self.tx = optax.adam(self.learning_rate)

        self.params = None
        self.batch_stats = None
        self.opt_state = None

        self.history: List[Dict[str, Any]] = []
        self.validation_predictions: pd.DataFrame = None

        self._prepare_data()

    def _construct_data_fast(
        self,
        n_samples,
        neg_reg_pool,
        neg_tgt_pool,
        emb_i_pos,
        emb_j_pos,
        y_pos,
        pos_reg_ids,
        pos_tgt_ids,
    ):
        """Constructs a dataset with positive and negative samples."""
        reg_ids_neg = np.random.choice(neg_reg_pool, size=n_samples, replace=True)
        tgt_ids_neg = np.random.choice(neg_tgt_pool, size=n_samples, replace=True)
        emb_i_neg = self.embeddings[reg_ids_neg]
        emb_j_neg = self.embeddings[tgt_ids_neg]
        y_neg = np.zeros(n_samples)

        emb_i = np.concatenate([emb_i_pos, emb_i_neg], axis=0)
        emb_j = np.concatenate([emb_j_pos, emb_j_neg], axis=0)
        y = np.concatenate([y_pos, y_neg], axis=0)
        reg_ids = np.concatenate([pos_reg_ids, reg_ids_neg], axis=0)
        tgt_ids = np.concatenate([pos_tgt_ids, tgt_ids_neg], axis=0)

        perm = np.random.permutation(len(y))
        return emb_i[perm], emb_j[perm], y[perm], reg_ids[perm], tgt_ids[perm]

    def _prepare_data(self):
        """Pre-computes data pools and the static validation set."""
        # Training data preparation
        self.pos_reg_ids = self.train_interactions["regulator_id"].values
        self.pos_tgt_ids = self.train_interactions["target_id"].values
        self.emb_i_pos_fixed = self.embeddings[self.pos_reg_ids]
        self.emb_j_pos_fixed = self.embeddings[self.pos_tgt_ids]
        self.y_pos_fixed = np.ones(len(self.train_interactions))
        self.neg_reg_pool = self.train_interactions["regulator_id"].unique()
        self.neg_tgt_pool = self.train_interactions["target_id"].unique()
        self.n_train = len(self.train_interactions)

        # Validation data preparation
        val_pos_reg_ids = self.validation_interactions["regulator_id"].values
        val_pos_tgt_ids = self.validation_interactions["target_id"].values
        val_emb_i_pos = self.embeddings[val_pos_reg_ids]
        val_emb_j_pos = self.embeddings[val_pos_tgt_ids]
        val_y_pos = np.ones(len(self.validation_interactions))
        val_neg_reg_pool = self.validation_interactions["regulator_id"].unique()
        val_neg_tgt_pool = self.validation_interactions["target_id"].unique()

        self.val_data = self._construct_data_fast(
            len(self.validation_interactions),
            val_neg_reg_pool,
            val_neg_tgt_pool,
            val_emb_i_pos,
            val_emb_j_pos,
            val_y_pos,
            val_pos_reg_ids,
            val_pos_tgt_ids,
        )

    def _init_model_state(self):
        """Initializes model parameters, batch stats, and optimizer state."""
        dummy_data = self._construct_data_fast(
            1,
            self.neg_reg_pool,
            self.neg_tgt_pool,
            self.emb_i_pos_fixed[:1],
            self.emb_j_pos_fixed[:1],
            self.y_pos_fixed[:1],
            self.pos_reg_ids[:1],
            self.pos_tgt_ids[:1],
        )

        variables = self.model.init(
            jax.random.PRNGKey(0), dummy_data[0], dummy_data[1], dummy_data[2], train=False
        )
        self.params = variables["params"]
        self.batch_stats = variables["batch_stats"]
        self.opt_state = self.tx.init(self.params)

    def train(self, max_epochs: int = 250):
        """
        Trains the model.

        Args:
            max_epochs: The number of epochs to train for.
        """
        if self.params is None:
            self._init_model_state()

        @jax.jit
        def train_step(params, batch_stats, opt_state, emb_i, emb_j, y):
            def loss_fn(params):
                (loss, updates) = self.model.apply(
                    {"params": params, "batch_stats": batch_stats},
                    emb_i,
                    emb_j,
                    y,
                    train=True,
                    mutable=["batch_stats"],
                )
                return jnp.mean(loss), updates["batch_stats"]

            (loss_val, new_batch_stats), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
            updates, new_opt_state = self.tx.update(grads, opt_state, params)
            new_params = optax.apply_updates(params, updates)
            return new_params, new_batch_stats, new_opt_state, loss_val

        @jax.jit
        def eval_step(params, batch_stats, emb_i, emb_j, y):
            loss = self.model.apply(
                {"params": params, "batch_stats": batch_stats}, emb_i, emb_j, y, train=False
            )
            return jnp.mean(loss)

        step = 0
        for epoch in range(max_epochs):
            train_data = self._construct_data_fast(
                self.n_train,
                self.neg_reg_pool,
                self.neg_tgt_pool,
                self.emb_i_pos_fixed,
                self.emb_j_pos_fixed,
                self.y_pos_fixed,
                self.pos_reg_ids,
                self.pos_tgt_ids,
            )
            train_losses = []

            for i in range(0, len(train_data[0]), self.batch_size):
                emb_i_batch, emb_j_batch, y_batch = (
                    train_data[0][i : i + self.batch_size],
                    train_data[1][i : i + self.batch_size],
                    train_data[2][i : i + self.batch_size],
                )
                self.params, self.batch_stats, self.opt_state, loss = train_step(
                    self.params, self.batch_stats, self.opt_state, emb_i_batch, emb_j_batch, y_batch
                )
                train_losses.append(loss)
                self.history.append(
                    {
                        "metric": loss.item(),
                        "metric_name": "loss",
                        "epoch": epoch,
                        "step": step,
                        "type": "train",
                    }
                )
                step += 1

            if epoch % 10 == 0:
                val_losses = []
                for i in range(0, len(self.val_data[0]), self.batch_size):
                    emb_i_batch, emb_j_batch, y_batch = (
                        self.val_data[0][i : i + self.batch_size],
                        self.val_data[1][i : i + self.batch_size],
                        self.val_data[2][i : i + self.batch_size],
                    )
                    loss = eval_step(
                        self.params, self.batch_stats, emb_i_batch, emb_j_batch, y_batch
                    )
                    val_losses.append(loss)

                mean_val_loss = np.mean(val_losses)
                self.history.append(
                    {"metric": mean_val_loss, "metric_name": "loss", "epoch": epoch, "type": "val"}
                )
                prauc = self.compute_prauc(self.val_data)
                self.history.append(
                    {"metric": prauc, "metric_name": "prauc", "epoch": epoch, "type": "val"}
                )
                print(
                    f"Epoch {epoch}, Train Loss: {np.mean(train_losses):.4f}, Val Loss: {mean_val_loss:.4f}, PRAUC: {prauc:.4f}"
                )

    def predict_dataset(self, data_tuple, batch_size: int = 4096) -> pd.DataFrame:
        """
        Makes predictions on a dataset.

        Args:
            data_tuple: A tuple containing embeddings, labels, and metadata.
            batch_size: The batch size for prediction.

        Returns:
            A DataFrame with predictions and metadata.
        """
        if self.params is None:
            raise RuntimeError("Model has not been trained. Call train() first.")

        @jax.jit
        def predict_batch(params, batch_stats, emb_i, emb_j):
            return self.model.apply(
                {"params": params, "batch_stats": batch_stats},
                emb_i,
                emb_j,
                train=False,
                method=MLP.predict,
            )

        emb_i_all, emb_j_all, y_all, reg_ids_all, tgt_ids_all = data_tuple

        all_logits = [
            np.array(
                predict_batch(
                    self.params,
                    self.batch_stats,
                    emb_i_all[i : i + batch_size],
                    emb_j_all[i : i + batch_size],
                )
            )
            for i in range(0, len(emb_i_all), batch_size)
        ]

        logits_flat = np.concatenate(all_logits)
        reg_names = [self.gene_names[i] for i in reg_ids_all]
        tgt_names = [self.gene_names[i] for i in tgt_ids_all]

        return pd.DataFrame(
            {"regulator": reg_names, "target": tgt_names, "score": logits_flat, "label": y_all}
        )

    def compute_prauc(self, data_tuple) -> float:
        """Computes the PRAUC for a given dataset."""
        df_preds = self.predict_dataset(data_tuple)
        prec, recall, _ = precision_recall_curve(df_preds["label"], df_preds["score"], pos_label=1)
        return auc(recall, prec)

    def get_history(self) -> pd.DataFrame:
        """Returns the training history as a DataFrame."""
        return pd.DataFrame(self.history)

    def get_validation_predictions(self) -> pd.DataFrame:
        """Returns predictions on the validation set as a DataFrame."""
        if self.validation_predictions is None:
            self.validation_predictions = self.predict_dataset(self.val_data)
        return self.validation_predictions

In [ ]:
model_llm = CausalPredictor(
    llm_embeddings,
    train_interactions,
    validation_interactions,
    gene_names,
)
model_llm.train(max_epochs=250)

In [ ]:
model = CausalPredictor(
    embeddings,
    train_interactions,
    validation_interactions,
    gene_names,
)
model.train(max_epochs=250)

In [ ]:
shared_embeddings = np.concatenate([embeddings, llm_embeddings], axis=1)
shared_model = CausalPredictor(
    shared_embeddings,
    train_interactions,
    validation_interactions,
    gene_names,
)
shared_model.train(max_epochs=250)

In [ ]:
def get_pr_curve(model, model_name, random_baseline=False):
    df_preds = model.predict_dataset(model.val_data)
    if random_baseline:
        prec, recall, _ = precision_recall_curve(
            df_preds["label"], np.random.rand(len(df_preds)), pos_label=1
        )
    else:
        prec, recall, _ = precision_recall_curve(df_preds["label"], df_preds["score"], pos_label=1)
    return pd.DataFrame({"precision": prec, "recall": recall, "model": model_name})

In [ ]:
df1 = get_pr_curve(model, "Evo2 features")
df2 = get_pr_curve(model_llm, "LLM features")
df3 = get_pr_curve(shared_model, "(Evo2+LLM) features")
df4 = get_pr_curve(model, "random choice", random_baseline=True)

plot_df = pd.concat([df1, df2, df3, df4])

In [ ]:
(
    gg.ggplot(plot_df)
    + gg.aes(x="recall", y="precision", color="model")
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.scale_y_continuous(limits=(0, 0.8))
)

# OLD

# Data Splitting

In [ ]:
def construct_data_fast(
    n_samples,
    neg_reg_pool,
    neg_tgt_pool,
    emb_i_pos,
    emb_j_pos,
    y_pos,
    pos_reg_ids,
    pos_tgt_ids,
):
    reg_ids_neg = np.random.choice(neg_reg_pool, size=n_samples, replace=True)
    tgt_ids_neg = np.random.choice(neg_tgt_pool, size=n_samples, replace=True)
    emb_i_neg = embeddings[reg_ids_neg]
    emb_j_neg = embeddings[tgt_ids_neg]
    y_neg = np.zeros(n_samples)

    emb_i = np.concatenate([emb_i_pos, emb_i_neg], axis=0)
    emb_j = np.concatenate([emb_j_pos, emb_j_neg], axis=0)
    y = np.concatenate([y_pos, y_neg], axis=0)
    reg_ids = np.concatenate([pos_reg_ids, reg_ids_neg], axis=0)
    tgt_ids = np.concatenate([pos_tgt_ids, tgt_ids_neg], axis=0)

    perm = np.random.permutation(len(y))
    return emb_i[perm], emb_j[perm], y[perm], reg_ids[perm], tgt_ids[perm]

In [ ]:
# Pre-compute positive samples (static)
pos_reg_ids = train_interactions["regulator_id"].values
pos_tgt_ids = train_interactions["target_id"].values
emb_i_pos_fixed = embeddings[pos_reg_ids]
emb_j_pos_fixed = embeddings[pos_tgt_ids]
y_pos_fixed = np.ones(len(train_interactions))

neg_reg_pool = train_interactions["regulator_id"].unique()
neg_tgt_pool = train_interactions["target_id"].unique()
n_train = len(train_interactions)


train_data_ = construct_data_fast(
    n_train,
    neg_reg_pool,
    neg_tgt_pool,
    emb_i_pos_fixed,
    emb_j_pos_fixed,
    y_pos_fixed,
    pos_reg_ids,
    pos_tgt_ids,
)

In [ ]:
# Pre-compute validation positives
val_pos_reg_ids = validation_interactions["regulator_id"].values
val_pos_tgt_ids = validation_interactions["target_id"].values
val_emb_i_pos = embeddings[val_pos_reg_ids]
val_emb_j_pos = embeddings[val_pos_tgt_ids]
val_y_pos = np.ones(len(validation_interactions))

# Pre-compute validation negative pools
val_neg_reg_pool = validation_interactions["regulator_id"].unique()
val_neg_tgt_pool = validation_interactions["target_id"].unique()

# Generate static validation set once (using the same fast function)
val_data = construct_data_fast(
    len(validation_interactions),
    val_neg_reg_pool,
    val_neg_tgt_pool,
    val_emb_i_pos,
    val_emb_j_pos,
    val_y_pos,
    val_pos_reg_ids,
    val_pos_tgt_ids,
)

# Model definition

In [ ]:
import jax.numpy as jnp
import jax
import flax.linen as nn
import optax


class MLP(nn.Module):
    hidden_dim: int

    @nn.compact
    def predict(self, emb_i, emb_j, train: bool):
        x = jnp.concatenate([emb_i, emb_j], axis=-1)
        x = nn.Dense(features=self.hidden_dim)(x)
        x = nn.BatchNorm(use_running_average=not train)(x)
        x = nn.relu(x)
        x = nn.Dense(features=1)(x)
        return x.squeeze(-1)

    @nn.compact
    def __call__(self, emb_i, emb_j, y, train: bool = True):
        y_pred = self.predict(emb_i, emb_j, train=train)
        loss = optax.sigmoid_binary_cross_entropy(y_pred, y)
        return loss

In [ ]:
@jax.jit
def predict_batch(params, batch_stats, emb_i, emb_j):
    return model.apply(
        {"params": params, "batch_stats": batch_stats},
        emb_i,
        emb_j,
        train=False,
        method=MLP.predict,
    )


def predict_dataset_with_meta(data_tuple, batch_size=4096):
    # Unpack all 5 elements returned by construct_data_fast
    emb_i_all, emb_j_all, y_all, reg_ids_all, tgt_ids_all = data_tuple

    all_logits = []

    for i in range(0, len(emb_i_all), batch_size):
        b_emb_i = emb_i_all[i : i + batch_size]
        b_emb_j = emb_j_all[i : i + batch_size]

        logits = predict_batch(params, batch_stats, b_emb_i, b_emb_j)
        all_logits.append(np.array(logits))

    logits_flat = np.concatenate(all_logits)

    # Convert integer IDs back to gene names
    reg_names = [gene_names[i] for i in reg_ids_all]
    tgt_names = [gene_names[i] for i in tgt_ids_all]

    return pd.DataFrame(
        {"regulator": reg_names, "target": tgt_names, "score": logits_flat, "label": y_all}
    )


def compute_prauc(data_tuple):
    df_preds = predict_dataset_with_meta(data_tuple)
    prec, recall, _ = precision_recall_curve(df_preds["label"], df_preds["score"], pos_label=1)
    return auc(recall, prec)


@jax.jit
def train_step(params, batch_stats, opt_state, emb_i, emb_j, y):
    def loss_fn(params):
        (loss, updates) = model.apply(
            {"params": params, "batch_stats": batch_stats},
            emb_i,
            emb_j,
            y,
            train=True,
            mutable=["batch_stats"],
        )
        return jnp.mean(loss), updates["batch_stats"]

    (loss_val, new_batch_stats), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
    updates, new_opt_state = tx.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    return new_params, new_batch_stats, new_opt_state, loss_val


@jax.jit
def eval_step(params, batch_stats, emb_i, emb_j, y):
    loss = model.apply({"params": params, "batch_stats": batch_stats}, emb_i, emb_j, y, train=False)
    return jnp.mean(loss)

In [ ]:
val_data = construct_data_fast(
    len(validation_interactions),
    val_neg_reg_pool,
    val_neg_tgt_pool,
    val_emb_i_pos,
    val_emb_j_pos,
    val_y_pos,
    val_pos_reg_ids,
    val_pos_tgt_ids,
)

In [ ]:
# max_epochs = int(1e3)
max_epochs = 250
learning_rate = 1e-3
batch_size = 1024


model = MLP(hidden_dim=4096)
variables = model.init(
    jax.random.PRNGKey(0), train_data_[0], train_data_[1], train_data_[2], train=False
)
tx = optax.adam(learning_rate)
opt_state = tx.init(variables["params"])
batch_stats = variables["batch_stats"]
params = variables["params"]
all_losses = []

step = 0
for epoch in range(max_epochs):
    train_data_ = construct_data_fast(
        n_train,
        neg_reg_pool,
        neg_tgt_pool,
        emb_i_pos_fixed,
        emb_j_pos_fixed,
        y_pos_fixed,
        pos_reg_ids,
        pos_tgt_ids,
    )
    train_losses = []

    for i in range(0, len(train_data_[0]), batch_size):
        emb_i_batch = train_data_[0][i : i + batch_size]
        emb_j_batch = train_data_[1][i : i + batch_size]
        y_batch = train_data_[2][i : i + batch_size]

        params, batch_stats, opt_state, loss = train_step(
            params, batch_stats, opt_state, emb_i_batch, emb_j_batch, y_batch
        )
        train_losses.append(loss)
        all_losses.append(
            {
                "metric": loss.item(),
                "metric_name": "loss",
                "epoch": epoch,
                "step": step,
                "type": "train",
            }
        )
        step += 1
    if epoch % 10 == 0:
        val_losses = []
        for i in range(0, len(val_data[0]), batch_size):
            emb_i_batch = val_data[0][i : i + batch_size]
            emb_j_batch = val_data[1][i : i + batch_size]
            y_batch = val_data[2][i : i + batch_size]

            loss = eval_step(params, batch_stats, emb_i_batch, emb_j_batch, y_batch)
            val_losses.append(loss)
        all_losses.append(
            {
                "metric": np.mean(val_losses),
                "metric_name": "loss",
                "epoch": epoch,
                "type": "val",
            }
        )

        prauc = compute_prauc(val_data)
        all_losses.append(
            {
                "metric": prauc,
                "metric_name": "prauc",
                "epoch": epoch,
                "type": "val",
            }
        )
        print(
            f"Epoch {epoch}, Train Loss: {np.mean(train_losses):.4f}, Val Loss: {np.mean(val_losses):.4f}, PRAUC: {prauc:.4f}"
        )
all_losses = pd.DataFrame(all_losses)

In [ ]:
(
    gg.ggplot(all_losses.query("type == 'val'").query("metric_name == 'prauc'"))
    + gg.aes(x="epoch", y="metric", color="type")
    + gg.geom_line()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(all_losses.query("type == 'val'").query("metric_name == 'loss'"))
    + gg.aes(x="epoch", y="metric", color="type")
    + gg.geom_line()
    + gg.theme_minimal()
)

In [ ]:
(
    gg.ggplot(all_losses.query("type == 'train'").query("metric_name == 'loss'"))
    + gg.aes(x="step", y="metric", color="type")
    + gg.geom_line()
    + gg.theme_minimal()
)

In [ ]:
df_train_preds = predict_dataset_with_meta(train_data_)
df_val_preds = predict_dataset_with_meta(val_data)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
df_train_preds["score"].hist(bins=100, ax=axes[0])
df_val_preds["score"].hist(bins=100, ax=axes[1])
plt.show()

# Evaluation

In [ ]:
prec, recall, _ = precision_recall_curve(df_val_preds["label"], df_val_preds["score"], pos_label=1)
plot_df1 = pd.DataFrame(
    {
        "precision": prec,
        "recall": recall,
        "model": "MLP on Evo2 features",
    }
)

prec, recall, _ = precision_recall_curve(
    df_val_preds["label"], np.random.rand(len(df_val_preds)), pos_label=1
)
plot_df2 = pd.DataFrame(
    {
        "precision": prec,
        "recall": recall,
        "model": "Random guess",
    }
)

plot_df = pd.concat([plot_df1, plot_df2])

fig = (
    gg.ggplot(plot_df)
    + gg.aes(x="recall", y="precision", color="model")
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.theme(figure_size=(5, 3))
    + gg.scale_y_continuous(limits=(0, 1))
)
fig

In [ ]:
df_val_preds.sort_values("score", ascending=False)

In [ ]:
# valid_regulators = [r for r in unique_regulators if r in gene_to_id]
# valid_targets = [t for t in validation_targets if t in gene_to_id]

valid_regulators = [r for r in heldout_regulators if r in gene_to_id]
valid_targets = [t for t in unique_targets if t in gene_to_id]

n_regs = len(valid_regulators)
n_tgts = len(valid_targets)
reg_ids = np.array([gene_to_id[r] for r in valid_regulators])
tgt_ids = np.array([gene_to_id[t] for t in valid_targets])

target_names = np.repeat(valid_targets, n_regs)
regulator_names = np.tile(valid_regulators, n_tgts)

emb_i_eval = embeddings[np.tile(reg_ids, n_tgts)]
emb_j_eval = embeddings[np.repeat(tgt_ids, n_regs)]

eval_batch_size = 4096
prediction_scores = []

for i in tqdm(range(0, len(emb_i_eval), eval_batch_size)):
    b_emb_i = emb_i_eval[i : i + eval_batch_size]
    b_emb_j = emb_j_eval[i : i + eval_batch_size]

    logits = predict_batch(params, batch_stats, b_emb_i, b_emb_j)
    prediction_scores.append(np.array(logits))

prediction_scores = np.concatenate(prediction_scores)

In [ ]:
val_pos_set = set(
    zip(validation_interactions["regulator_gene"], validation_interactions["target_gene"])
)

gt_labels = np.array(
    [1 if (r, t) in val_pos_set else 0 for r, t in zip(regulator_names, target_names)]
)

In [ ]:
res_ = pd.DataFrame(
    {
        "regulator_name": regulator_names,
        "target_name": target_names,
        "prediction_score": prediction_scores,
        "gt_label": gt_labels,
    }
)

In [ ]:
res_sorted = res_.sort_values("prediction_score", ascending=False).reset_index(drop=True)
precision_manual = np.cumsum(res_sorted["gt_label"]) / np.arange(1, len(res_sorted) + 1)

In [ ]:
prec, recall, _ = precision_recall_curve(gt_labels, prediction_scores, pos_label=1)

plot_df1 = pd.DataFrame(
    {
        "precision": prec,
        "recall": recall,
        "model": "MLP on Evo2 features",
    }
)

prec, recall, _ = precision_recall_curve(gt_labels, np.random.rand(len(gt_labels)), pos_label=1)
prop_of_positives = gt_labels.mean()
plot_df2 = pd.DataFrame(
    {
        "precision": prec,
        "recall": recall,
        "model": "Random guess",
    }
)

plot_df = pd.concat([plot_df1, plot_df2])

fig = (
    gg.ggplot(plot_df)
    + gg.aes(x="recall", y="precision", color="model")
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.theme(figure_size=(5, 3))
    + gg.geom_hline(yintercept=prop_of_positives, color="black", linetype="dashed")
    + gg.scale_y_continuous(limits=(0, 1))
    + gg.scale_x_continuous(limits=(0, 0.25))
)
fig